In [ ]:
from pathlib import Path
import pandas as pd
import sys
# Resolve project root directory
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "scaled"


import importlib
import src.sequence_creation as sequence_creation

importlib.reload(sequence_creation)

from src.sequence_creation import (
    MODEL_FEATURES,
    MISSINGNESS_FEATURE,
    SCALED_FEATURES,
    LSTM_FEATURES,
    validate_sequence_input,
    create_lstm_sequences,
    print_sequence_report,
    save_sequences,
)

In [ ]:
# ============================================================
# SEQUENCE CONFIGURATION
# ============================================================

SEQUENCE_LENGTH = 20

# Allow normal sampling jitter and timing jitter.
# A gap > 60 seconds becomes a sequence boundary.
MAX_ALLOWED_GAP_SECONDS = 60.0

TARGET_MODE = "binary"

LABEL_COLUMN = "tipo_anomalia"

print("Sequence length:", SEQUENCE_LENGTH)
print(
    "Maximum allowed timestamp gap:",
    MAX_ALLOWED_GAP_SECONDS,
    "seconds",
)
print("Target mode:", TARGET_MODE)
print("Number of model features:", len(MODEL_FEATURES))

In [ ]:
# ============================================================
# LOAD TRAIN
# ============================================================

train_path = (
    PROCESSED_DIR /
    "train_features_scaled.parquet"
)

df_train_scaled = pd.read_parquet(
    train_path
)

print(
    "Train dataset loaded:",
    df_train_scaled.shape
)

In [ ]:
# ============================================================
# LOAD VALIDATION
# ============================================================

validation_path = (
    PROCESSED_DIR /
    "validation_features_scaled.parquet"
)

df_validation_scaled = pd.read_parquet(
    validation_path
)

print(
    "Validation dataset loaded:",
    df_validation_scaled.shape
)

In [ ]:
# ============================================================
# LOAD TEST
# ============================================================

test_path = (
    PROCESSED_DIR /
    "test_features_scaled.parquet"
)

df_test_scaled = pd.read_parquet(
    test_path
)

print(
    "Test dataset loaded:",
    df_test_scaled.shape
)

In [ ]:
# ============================================================
# VERIFY SCALED FEATURES
# ============================================================

print("=" * 80)
print("SCALED FEATURE CHECK")
print("=" * 80)

print("\nExpected scaled features:")

for feature in SCALED_FEATURES:
    print("  ✓", feature)

print(
    "\nNumber of scaled features:",
    len(SCALED_FEATURES),
)

for name, df in [
    ("TRAIN", df_train_scaled),
    ("VALIDATION", df_validation_scaled),
    ("TEST", df_test_scaled),
]:

    missing = [
        col
        for col in SCALED_FEATURES
        if col not in df.columns
    ]

    print(f"\n{name}:")

    if missing:
        print("Missing:", missing)
    else:
        print("All 18 scaled features present.")

In [ ]:
# ============================================================
# VERIFY NO MODEL NaNs
# ============================================================

for name, df in [
    ("TRAIN", df_train_scaled),
    ("VALIDATION", df_validation_scaled),
    ("TEST", df_test_scaled),
]:

    nan_cells = int(
        df[SCALED_FEATURES]
        .isna()
        .sum()
        .sum()
    )

    nan_rows = int(
        df[SCALED_FEATURES]
        .isna()
        .any(axis=1)
        .sum()
    )

    print("=" * 80)
    print(name)
    print("=" * 80)

    print("NaN cells:", nan_cells)
    print("Rows containing NaN:", nan_rows)

In [ ]:
# ============================================================
# SEQUENCE BOUNDARY CHECK
# ============================================================

for name, df in [
    ("TRAIN", df_train_scaled),
    ("VALIDATION", df_validation_scaled),
    ("TEST", df_test_scaled),
]:

    print("=" * 80)
    print(f"{name} — SEQUENCE BOUNDARIES")
    print("=" * 80)

    print("\nsequence_unsafe:")
    print(
        df["sequence_unsafe"]
        .fillna(False)
        .astype(bool)
        .value_counts()
    )

    print("\nGap category:")
    print(
        df["gap_category"]
        .value_counts(dropna=False)
    )

In [ ]:
# ============================================================
# VALIDATE SEQUENCE INPUTS
# ============================================================

validate_sequence_input(
    df_train_scaled,
    scaled_features=SCALED_FEATURES,
)

validate_sequence_input(
    df_validation_scaled,
    scaled_features=SCALED_FEATURES,
)

validate_sequence_input(
    df_test_scaled,
    scaled_features=SCALED_FEATURES,
)

print("✓ Train input valid")
print("✓ Validation input valid")
print("✓ Test input valid")

In [ ]:
# ============================================================
# CREATE TRAIN SEQUENCES
# ============================================================

X_train, y_train, train_metadata, train_report = (
    create_lstm_sequences(
        df_train_scaled,
        sequence_length=SEQUENCE_LENGTH,
        scaled_features=SCALED_FEATURES,
        label_column="tipo_anomalia",
        max_allowed_gap_seconds=MAX_ALLOWED_GAP_SECONDS,
        create_targets=True,
    )
)

print_sequence_report(
    train_report,
    dataset_name="TRAIN",
)

In [ ]:
# ============================================================
# CREATE VALIDATION SEQUENCES
# ============================================================

X_validation, y_validation, validation_metadata, validation_report = (
    create_lstm_sequences(
        df_validation_scaled,
        sequence_length=SEQUENCE_LENGTH,
        scaled_features=SCALED_FEATURES,
        max_allowed_gap_seconds=MAX_ALLOWED_GAP_SECONDS,
        create_targets=True,
    )
)

print_sequence_report(
    validation_report,
    dataset_name="VALIDATION",
)

In [ ]:
# ============================================================
# CREATE TEST SEQUENCES
# ============================================================

X_test, y_test, test_metadata, test_report = (
    create_lstm_sequences(
        df_test_scaled,
        sequence_length=SEQUENCE_LENGTH,
        scaled_features=SCALED_FEATURES,
        max_allowed_gap_seconds=MAX_ALLOWED_GAP_SECONDS,
        create_targets=True,
    )
)

print_sequence_report(
    test_report,
    dataset_name="TEST",
)

In [ ]:
# ============================================================
# FINAL SEQUENCE SHAPES
# ============================================================

print("=" * 80)
print("LSTM SEQUENCE SHAPES")
print("=" * 80)

print("\nTRAIN")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nVALIDATION")
print("X_validation:", X_validation.shape)

print("\nTEST")
print("X_test:", X_test.shape)


In [ ]:
# ============================================================
# SAVE LSTM SEQUENCES
# ============================================================

PROCESSED_DIR = PROJECT_ROOT / "data" / "sequences"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------
save_sequences(
    X=X_train,
    y=y_train,
    metadata=train_metadata,
    output_prefix=str(PROCESSED_DIR / "train_lstm_sequences"),
)

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------
save_sequences(
    X=X_validation,
    y=y_validation,
    metadata=validation_metadata,
    output_prefix=str(PROCESSED_DIR / "validation_lstm_sequences"),
)

# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------
save_sequences(
    X=X_test,
    y=y_test,
    metadata=test_metadata,
    output_prefix=str(PROCESSED_DIR / "test_lstm_sequences"),
)

print("=" * 70)
print("LSTM SEQUENCES SAVED")
print("=" * 70)

print("\nFiles:")

print(f"  {PROCESSED_DIR / 'train_lstm_sequences_X.npy'}")
print(f"  {PROCESSED_DIR / 'train_lstm_sequences_y.npy'}")
print(f"  {PROCESSED_DIR / 'train_lstm_sequences_metadata.parquet'}")

print(f"  {PROCESSED_DIR / 'validation_lstm_sequences_X.npy'}")
print(f"  {PROCESSED_DIR / 'validation_lstm_sequences_y.npy'}")
print(f"  {PROCESSED_DIR / 'validation_lstm_sequences_metadata.parquet'}")

print(f"  {PROCESSED_DIR / 'test_lstm_sequences_X.npy'}")
print(f"  {PROCESSED_DIR / 'test_lstm_sequences_y.npy'}")
print(f"  {PROCESSED_DIR / 'test_lstm_sequences_metadata.parquet'}")

In [ ]:
import numpy as np

print("=" * 70)
print("LSTM SEQUENCE SHAPES AND TARGET DISTRIBUTION")
print("=" * 70)

# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------
print("\nTRAIN")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("Unique y_train values:")
print(np.unique(y_train, return_counts=True))

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------
print("\nVALIDATION")
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

print("Unique y_validation values:")
print(np.unique(y_validation, return_counts=True))

# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------
print("\nTEST")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("Unique y_test values:")
print(np.unique(y_test, return_counts=True))